
# 4.2. El módulo `unittest`

El módulo integrado `unittest` proporciona un framework para escribir pruebas unitarias en Python.

Una prueba unitaria permite comprobar que una parte pequeña del código funciona como esperamos.

En este cuaderno trabajaremos en dos niveles:

1. Primero, con un ejemplo sencillo de funciones de texto.
2. Después, aplicaremos `unittest` a la aplicación `AppTickets`.

La intención es que el estudiante entienda no solo la sintaxis, sino también el valor práctico de las pruebas unitarias en una aplicación orientada a objetos.



## 1. ¿Qué es una prueba unitaria?

Una prueba unitaria verifica una unidad pequeña de código.

Esa unidad puede ser:

- una función;
- un método;
- una validación;
- una propiedad;
- una regla de negocio.

Por ejemplo, si tenemos una función que debe unir textos, una prueba unitaria puede confirmar que la función devuelve exactamente el resultado esperado.

La idea básica es:

> Si sabemos qué debe devolver una función, podemos escribir una prueba para comprobarlo automáticamente.



## 2. Pasos mínimos para usar `unittest`

Para probar código con `unittest`, normalmente seguimos estos pasos:

1. Importar el módulo `unittest`.
2. Crear una clase que herede de `unittest.TestCase`.
3. Escribir métodos de prueba.
4. Nombrar los métodos iniciando con `test`.
5. Usar métodos de aserción como `assertEqual`, `assertTrue`, `assertFalse` o `assertRaises`.
6. Ejecutar las pruebas.

En archivos `.py`, normalmente se usa:

```python
if __name__ == "__main__":
    unittest.main()
```

En Jupyter Notebook usaremos una forma ligeramente distinta para evitar conflictos con el kernel.



## 3. Importar módulos necesarios


In [1]:

import unittest
from abc import ABC, abstractmethod
from functools import wraps
from datetime import datetime
import re
import os



## 4. Ejemplo base: tres funciones de texto

Comenzaremos con tres funciones:

- `prepend(s, c)`: agrega texto al inicio.
- `append(s, c)`: agrega texto al final.
- `insert(s, c, pos)`: inserta texto en una posición específica.

La tercera función tendrá un error intencional para demostrar cómo `unittest` detecta fallas.


In [2]:

def prepend(s, c):
    return c + s


def append(s, c):
    return s + c


def insert_con_error(s, c, pos):
    # Esta función contiene un error intencional.
    # El segmento s[pos:-1] excluye el último carácter de la cadena.
    return s[0:pos] + c + s[pos:-1]



## 5. Crear una clase de prueba

La clase debe heredar de `unittest.TestCase`.

Cada método de prueba debe iniciar con `test`.

Si un método no inicia con `test`, `unittest` no lo ejecutará automáticamente como prueba.


In [3]:

class TestFuncionesTextoConError(unittest.TestCase):

    def test_prepend(self):
        self.assertEqual(prepend("bar", "foo"), "foobar")

    def test_append(self):
        self.assertEqual(append("bar", "foo"), "barfoo")

    def test_insert(self):
        self.assertEqual(insert_con_error("wetor", "buca", 2), "webucator")



## 6. Ejecutar pruebas dentro de Jupyter Notebook

En un archivo `.py` normalmente usaríamos:

```python
if __name__ == "__main__":
    unittest.main()
```

Pero en Jupyter Notebook es mejor construir el conjunto de pruebas manualmente con:

```python
suite = unittest.TestLoader().loadTestsFromTestCase(...)
runner = unittest.TextTestRunner(...)
runner.run(suite)
```

Esto evita que `unittest.main()` intente interpretar argumentos internos del notebook.


In [4]:

suite = unittest.TestLoader().loadTestsFromTestCase(TestFuncionesTextoConError)
runner = unittest.TextTestRunner(verbosity=2)
resultado = runner.run(suite)


test_append (__main__.TestFuncionesTextoConError.test_append) ... ok
test_insert (__main__.TestFuncionesTextoConError.test_insert) ... FAIL
test_prepend (__main__.TestFuncionesTextoConError.test_prepend) ... ok

FAIL: test_insert (__main__.TestFuncionesTextoConError.test_insert)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\Victor\AppData\Local\Temp\ipykernel_27080\1320880613.py", line 10, in test_insert
    self.assertEqual(insert_con_error("wetor", "buca", 2), "webucator")
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 'webucato' != 'webucator'
- webucato
+ webucator
?         +


----------------------------------------------------------------------
Ran 3 tests in 0.010s

FAILED (failures=1)



## 7. Interpretar el resultado de una prueba fallida

La prueba de `insert_con_error()` falla porque esperábamos:

```python
webucator
```

pero la función devuelve otro resultado.

El problema está en esta parte:

```python
s[pos:-1]
```

Ese corte toma la cadena desde `pos`, pero se detiene antes del último carácter.  
Por eso se pierde la última letra.

La versión correcta debe usar:

```python
s[pos:]
```


In [ ]:

def insert(s, c, pos):
    # Versión corregida.
    return s[0:pos] + c + s[pos:]


class TestFuncionesTextoCorregidas(unittest.TestCase):

    def test_prepend(self):
        self.assertEqual(prepend("bar", "foo"), "foobar")

    def test_append(self):
        self.assertEqual(append("bar", "foo"), "barfoo")

    def test_insert(self):
        self.assertEqual(insert("wetor", "buca", 2), "webucator")


suite = unittest.TestLoader().loadTestsFromTestCase(TestFuncionesTextoCorregidas)
runner = unittest.TextTestRunner(verbosity=2)
resultado = runner.run(suite)



## 8. Métodos de aserción comunes

`unittest` incluye varios métodos para comprobar resultados.

| Método | Uso |
|---|---|
| `assertEqual(a, b)` | Verifica que `a` sea igual a `b` |
| `assertNotEqual(a, b)` | Verifica que `a` sea diferente de `b` |
| `assertTrue(x)` | Verifica que `x` sea verdadero |
| `assertFalse(x)` | Verifica que `x` sea falso |
| `assertIsInstance(obj, clase)` | Verifica que un objeto sea instancia de una clase |
| `assertRaises(Error)` | Verifica que se lance un error esperado |

Ahora aplicaremos estas ideas a la aplicación de tickets.



## 9. Aplicación base: AppTickets

La siguiente celda contiene la aplicación de tickets que usaremos para las pruebas.

Se mantiene en el notebook para que el cuaderno pueda ejecutarse sin depender de archivos externos.


In [5]:

def registrar_accion(funcion):
    @wraps(funcion)
    def envoltura(*args, **kwargs):
        objeto = args[0]

        fecha_hora = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        clase = objeto.__class__.__name__
        metodo = funcion.__name__
        folio = getattr(objeto, "folio", "N/A")
        estado_anterior = getattr(objeto, "estado", "N/A")

        resultado = funcion(*args, **kwargs)

        estado_nuevo = getattr(objeto, "estado", "N/A")

        registro = {
            "fecha_hora": fecha_hora,
            "clase": clase,
            "folio": folio,
            "metodo": metodo,
            "estado_anterior": estado_anterior,
            "estado_nuevo": estado_nuevo
        }

        if hasattr(objeto, "_historial"):
            objeto._historial.append(registro)

        with open("registro_acciones.log", "a", encoding="utf-8") as archivo:
            archivo.write(
                f"{fecha_hora} | "
                f"{clase} | "
                f"{folio} | "
                f"{metodo} | "
                f"{estado_anterior} -> {estado_nuevo}\n"
            )

        return resultado

    return envoltura


class TicketBase(ABC):

    @abstractmethod
    def atender(self):
        pass


class Ticket(TicketBase):

    ESTADOS_VALIDOS = ["Abierto", "En proceso", "Cerrado"]
    _contador_global = 0

    def __init__(self, folio, titulo, descripcion):
        self._historial = []

        self.folio = folio
        self.titulo = titulo
        self.descripcion = descripcion
        self.estado = "Abierto"

        Ticket._contador_global += 1

        self._registrar_evento_manual("crear", "N/A", self.estado)

    @property
    def folio(self):
        return self._folio

    @folio.setter
    def folio(self, nuevo_folio):
        if not self.es_folio_valido(nuevo_folio):
            raise ValueError(f"Folio no válido: {nuevo_folio}")

        self._folio = nuevo_folio

    @property
    def titulo(self):
        return self._titulo

    @titulo.setter
    def titulo(self, nuevo_titulo):
        self._titulo = nuevo_titulo

    @property
    def descripcion(self):
        return self._descripcion

    @descripcion.setter
    def descripcion(self, nueva_descripcion):
        self._descripcion = nueva_descripcion

    @property
    def estado(self):
        return self._estado

    @estado.setter
    def estado(self, nuevo_estado):
        if nuevo_estado not in self.ESTADOS_VALIDOS:
            raise ValueError(f"Estado no válido: {nuevo_estado}")

        self._estado = nuevo_estado

    @property
    def historial(self):
        return tuple(self._historial)

    def _registrar_evento_manual(self, metodo, estado_anterior, estado_nuevo):
        registro = {
            "fecha_hora": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "clase": self.__class__.__name__,
            "folio": self.folio,
            "metodo": metodo,
            "estado_anterior": estado_anterior,
            "estado_nuevo": estado_nuevo
        }

        self._historial.append(registro)

    @registrar_accion
    def cambiar_estado(self, nuevo_estado):
        self.estado = nuevo_estado

    @registrar_accion
    def cerrar(self):
        self.estado = "Cerrado"

    @classmethod
    def total_tickets_creados(cls):
        return cls._contador_global

    @staticmethod
    def es_folio_valido(folio):
        return bool(re.fullmatch(r"^TK-?\d{3}$", folio))

    def registrar_atencion(self):
        self.estado = "En proceso"
        return f"Ticket {self.folio} marcado como En proceso."

    @registrar_accion
    def atender(self):
        return self.registrar_atencion()

    def __str__(self):
        return (
            f"{self.__class__.__name__} | "
            f"Folio: {self.folio} | "
            f"Título: {self.titulo} | "
            f"Estado: {self.estado}"
        )


class TicketSoporte(Ticket):

    @registrar_accion
    def atender(self):
        mensaje_base = super().atender()
        return (
            f"{mensaje_base} "
            f"El ticket de soporte {self.folio} está siendo atendido por mesa de ayuda."
        )


class TicketDesarrollo(Ticket):

    @registrar_accion
    def atender(self):
        mensaje_base = super().atender()
        return (
            f"{mensaje_base} "
            f"El ticket de desarrollo {self.folio} fue asignado al equipo de programación."
        )


class TicketInfraestructura(Ticket):

    @registrar_accion
    def atender(self):
        mensaje_base = super().atender()
        return (
            f"{mensaje_base} "
            f"El ticket de infraestructura {self.folio} fue enviado al área de servidores/redes."
        )



## 10. Pruebas para validación de folios

La primera regla que probaremos será la validación de folios.

De acuerdo con la expresión regular:

```python
r"^TK-?\d{3}$"
```

Son válidos:

- `TK-001`
- `TK001`

No son válidos:

- `TK-01`
- `ABC-001`
- `TK-1000`


In [6]:

class TestValidacionFolios(unittest.TestCase):

    def test_folio_con_guion_es_valido(self):
        self.assertTrue(Ticket.es_folio_valido("TK-001"))

    def test_folio_sin_guion_es_valido(self):
        self.assertTrue(Ticket.es_folio_valido("TK001"))

    def test_folio_con_dos_digitos_no_es_valido(self):
        self.assertFalse(Ticket.es_folio_valido("TK-01"))

    def test_folio_con_prefijo_incorrecto_no_es_valido(self):
        self.assertFalse(Ticket.es_folio_valido("ABC-001"))

    def test_folio_con_cuatro_digitos_no_es_valido(self):
        self.assertFalse(Ticket.es_folio_valido("TK-1000"))


suite = unittest.TestLoader().loadTestsFromTestCase(TestValidacionFolios)
runner = unittest.TextTestRunner(verbosity=2)
resultado = runner.run(suite)


test_folio_con_cuatro_digitos_no_es_valido (__main__.TestValidacionFolios.test_folio_con_cuatro_digitos_no_es_valido) ... ok
test_folio_con_dos_digitos_no_es_valido (__main__.TestValidacionFolios.test_folio_con_dos_digitos_no_es_valido) ... ok
test_folio_con_guion_es_valido (__main__.TestValidacionFolios.test_folio_con_guion_es_valido) ... ok
test_folio_con_prefijo_incorrecto_no_es_valido (__main__.TestValidacionFolios.test_folio_con_prefijo_incorrecto_no_es_valido) ... ok
test_folio_sin_guion_es_valido (__main__.TestValidacionFolios.test_folio_sin_guion_es_valido) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.013s

OK



## 11. Probar creación de tickets

Ahora probaremos que un ticket se cree con sus valores iniciales correctos.

Validaremos:

- folio;
- título;
- descripción;
- estado inicial;
- historial inicial.


In [7]:

class TestCreacionTicket(unittest.TestCase):

    def test_ticket_se_crea_con_estado_abierto(self):
        ticket = TicketSoporte(
            "TK-010",
            "Error de acceso",
            "El usuario no puede entrar al sistema."
        )

        self.assertEqual(ticket.folio, "TK-010")
        self.assertEqual(ticket.titulo, "Error de acceso")
        self.assertEqual(ticket.descripcion, "El usuario no puede entrar al sistema.")
        self.assertEqual(ticket.estado, "Abierto")

    def test_ticket_tiene_historial_inicial(self):
        ticket = TicketSoporte(
            "TK-011",
            "Restablecer contraseña",
            "El usuario olvidó su contraseña."
        )

        self.assertEqual(len(ticket.historial), 1)
        self.assertEqual(ticket.historial[0]["metodo"], "crear")


suite = unittest.TestLoader().loadTestsFromTestCase(TestCreacionTicket)
runner = unittest.TextTestRunner(verbosity=2)
resultado = runner.run(suite)


test_ticket_se_crea_con_estado_abierto (__main__.TestCreacionTicket.test_ticket_se_crea_con_estado_abierto) ... ok
test_ticket_tiene_historial_inicial (__main__.TestCreacionTicket.test_ticket_tiene_historial_inicial) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.005s

OK



## 12. Probar errores esperados con `assertRaises`

Una buena prueba no solo confirma casos correctos.

También debe confirmar que el programa rechaza datos incorrectos.

Para eso usamos `assertRaises`.

En este caso, si intentamos crear un ticket con folio inválido, debe generarse un `ValueError`.


In [10]:

class TestErroresEsperados(unittest.TestCase):

    def test_folio_invalido_lanza_value_error(self):
        with self.assertRaises(ValueError):
            TicketSoporte(
                "ABC-123",
                "Folio incorrecto",
                "Este ticket no debe crearse."
            )

    def test_estado_invalido_lanza_value_error(self):
        ticket = TicketSoporte(
            "TK-020",
            "Estado incorrecto",
            "Se probará un estado inválido."
        )

        with self.assertRaises(ValueError):
            ticket.estado = "Pendiente"


suite = unittest.TestLoader().loadTestsFromTestCase(TestErroresEsperados)
runner = unittest.TextTestRunner(verbosity=2)
resultado = runner.run(suite)


test_estado_invalido_lanza_value_error (__main__.TestErroresEsperados.test_estado_invalido_lanza_value_error) ... ok
test_folio_invalido_lanza_value_error (__main__.TestErroresEsperados.test_folio_invalido_lanza_value_error) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.006s

OK



## 13. Probar métodos de negocio

Ahora probaremos métodos que cambian el estado del ticket.

Validaremos:

- `atender()`: debe cambiar el estado a `En proceso`.
- `cerrar()`: debe cambiar el estado a `Cerrado`.

Como estos métodos están decorados, también generan registros en el historial y escriben en el archivo `registro_acciones.log`.


In [ ]:

def limpiar_archivo_log():
    if os.path.exists("registro_acciones.log"):
        os.remove("registro_acciones.log")


class TestMetodosNegocio(unittest.TestCase):

    def setUp(self):
        limpiar_archivo_log()

    def test_atender_cambia_estado_a_en_proceso(self):
        ticket = TicketSoporte(
            "TK-030",
            "Error de sistema",
            "Se requiere revisar la aplicación."
        )

        respuesta = ticket.atender()

        self.assertEqual(ticket.estado, "En proceso")
        self.assertIn("mesa de ayuda", respuesta)

    def test_cerrar_cambia_estado_a_cerrado(self):
        ticket = TicketDesarrollo(
            "TK-031",
            "Error en reporte",
            "El reporte no genera PDF."
        )

        ticket.cerrar()

        self.assertEqual(ticket.estado, "Cerrado")


suite = unittest.TestLoader().loadTestsFromTestCase(TestMetodosNegocio)
runner = unittest.TextTestRunner(verbosity=2)
resultado = runner.run(suite)



## 14. Probar herencia y polimorfismo

La aplicación tiene diferentes tipos de ticket:

- `TicketSoporte`
- `TicketDesarrollo`
- `TicketInfraestructura`

Todos heredan de `Ticket`, pero cada uno implementa su propia versión de `atender()`.

Esto permite probar polimorfismo: mismo método, diferente comportamiento.


In [ ]:

class TestPolimorfismoTickets(unittest.TestCase):

    def setUp(self):
        limpiar_archivo_log()

    def test_ticket_soporte_responde_como_soporte(self):
        ticket = TicketSoporte("TK-040", "Soporte", "Caso de soporte.")
        respuesta = ticket.atender()

        self.assertIn("mesa de ayuda", respuesta)

    def test_ticket_desarrollo_responde_como_desarrollo(self):
        ticket = TicketDesarrollo("TK-041", "Desarrollo", "Caso de desarrollo.")
        respuesta = ticket.atender()

        self.assertIn("programación", respuesta)

    def test_ticket_infraestructura_responde_como_infraestructura(self):
        ticket = TicketInfraestructura("TK-042", "Infraestructura", "Caso de infraestructura.")
        respuesta = ticket.atender()

        self.assertIn("servidores/redes", respuesta)


suite = unittest.TestLoader().loadTestsFromTestCase(TestPolimorfismoTickets)
runner = unittest.TextTestRunner(verbosity=2)
resultado = runner.run(suite)



## 15. Probar que el decorador registra historial

El decorador `registrar_accion` no solo ejecuta el método original.

También agrega una entrada al historial del ticket.

Probaremos que al atender un ticket el historial aumenta.


In [ ]:

class TestHistorialDecorador(unittest.TestCase):

    def setUp(self):
        limpiar_archivo_log()

    def test_atender_agrega_eventos_al_historial(self):
        ticket = TicketSoporte(
            "TK-050",
            "Historial",
            "Se probará el historial."
        )

        historial_inicial = len(ticket.historial)

        ticket.atender()

        historial_final = len(ticket.historial)

        self.assertGreater(historial_final, historial_inicial)

    def test_cerrar_registra_estado_anterior_y_nuevo(self):
        ticket = TicketSoporte(
            "TK-051",
            "Cerrar",
            "Se probará cierre."
        )

        ticket.cerrar()

        ultimo_evento = ticket.historial[-1]

        self.assertEqual(ultimo_evento["estado_anterior"], "Abierto")
        self.assertEqual(ultimo_evento["estado_nuevo"], "Cerrado")


suite = unittest.TestLoader().loadTestsFromTestCase(TestHistorialDecorador)
runner = unittest.TextTestRunner(verbosity=2)
resultado = runner.run(suite)



## 16. Ejecutar varias clases de prueba juntas

Hasta ahora ejecutamos cada clase por separado.

También podemos agrupar varias clases en un solo conjunto de pruebas.

Esto permite probar la aplicación completa de forma más ordenada.


In [ ]:

suite_general = unittest.TestSuite()

for clase_prueba in [
    TestValidacionFolios,
    TestCreacionTicket,
    TestErroresEsperados,
    TestMetodosNegocio,
    TestPolimorfismoTickets,
    TestHistorialDecorador
]:
    pruebas = unittest.TestLoader().loadTestsFromTestCase(clase_prueba)
    suite_general.addTests(pruebas)

runner = unittest.TextTestRunner(verbosity=2)
resultado_general = runner.run(suite_general)

print()
print("Resumen:")
print("Pruebas ejecutadas:", resultado_general.testsRun)
print("Errores:", len(resultado_general.errors))
print("Fallos:", len(resultado_general.failures))
print("¿Todo correcto?:", resultado_general.wasSuccessful())



## 17. Diferencia entre error y fallo

En `unittest`, no todo problema significa lo mismo.

| Resultado | Significado |
|---|---|
| Failure / Fallo | La prueba se ejecutó, pero el resultado no fue el esperado |
| Error | La prueba no pudo completarse por una excepción no controlada |

Ejemplo:

- Si esperábamos `"webucator"` y recibimos `"webucato"`, es un fallo.
- Si el código intenta usar una variable inexistente, es un error.

Esta diferencia ayuda a diagnosticar mejor el problema.



## 18. Estructura recomendada para archivos `.py`

En un proyecto real, normalmente tendríamos los archivos separados.

Por ejemplo:

```text
proyecto/
│
├── app_tickets.py
└── test_app_tickets.py
```

El archivo `test_app_tickets.py` podría tener una estructura como esta:

```python
import unittest
from app_tickets import TicketSoporte

class TestTicketSoporte(unittest.TestCase):

    def test_ticket_se_crea_abierto(self):
        ticket = TicketSoporte("TK-001", "Soporte", "Descripción")
        self.assertEqual(ticket.estado, "Abierto")

if __name__ == "__main__":
    unittest.main()
```

Esto permite ejecutar las pruebas desde terminal con:

```bash
python test_app_tickets.py
```



## 19. Buenas prácticas al escribir pruebas unitarias

Al escribir pruebas con `unittest`, conviene seguir estas recomendaciones:

1. Cada prueba debe comprobar una idea específica.
2. El nombre del método debe explicar qué se está probando.
3. Las pruebas deben incluir casos correctos e incorrectos.
4. Los errores esperados deben probarse con `assertRaises`.
5. Las pruebas no deben depender innecesariamente de otras pruebas.
6. Si una prueba genera archivos, conviene limpiarlos antes o después.
7. Las pruebas deben ser fáciles de leer.

Una buena prueba no solo detecta errores.  
También documenta cómo debería comportarse el código.



## 20. Conclusión didáctica

`unittest` permite verificar automáticamente que nuestro código se comporta como esperamos.

En este cuaderno vimos que puede utilizarse para probar:

- funciones simples;
- errores intencionales;
- validaciones;
- creación de objetos;
- propiedades;
- métodos de negocio;
- herencia;
- polimorfismo;
- decoradores;
- historial;
- errores esperados.

Frase para clase:

> Una prueba unitaria convierte una expectativa en una comprobación automática.
